# F2
Lucas Andersen (pbp8nq@virginia.edu​) 

DS 5001

May 8, 2026

## Setup

In [109]:
import pandas as pd
import numpy as np
import re
import os
import nltk

F1_path = 'data/F1'
F2_path = 'data/F2'
os.makedirs(F2_path, exist_ok = True)

In [110]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)

True

In [111]:
OHCO = ['work_id', 'part_num','chap_num','para_num','sent_num','token_num']

## Load F1 Data

In [112]:
LIB = pd.read_csv(f"{F1_path}/LIB.csv", index_col = OHCO[0])
DOC = pd.read_csv(f"{F1_path}/DOC.csv", index_col = OHCO[0:4])

print(f"LIB: {len(LIB)} works")
print(f"DOC: {len(DOC)} paragraphs")

DOC.head()

LIB: 49 works
DOC: 3579 paragraphs


para_str
work_id         part_num chap_num para_num                                                   
modest_proposal 1        1        1         It is a melancholy object to those, who walk t...
                                  2         I think it is agreed by all parties, that this...
                                  3         But my intention is very far from being confin...
                                  4         As to my own part, having turned my thoughts f...
                                  5         There is likewise another great advantage in m...

## Tokenize and Annotate

In [113]:
def tokenize(doc_df, OHCO=OHCO, remove_pos_tuple=False, ws=False):
    
    # Paragraphs to sentences
    df = doc_df.para_str.apply(lambda x: pd.Series(nltk.sent_tokenize(x))).stack().to_frame().rename(columns = {0:'sent_str'})

    # Sentences to tokens
    def word_tokenize(x):
        if ws:
            s = pd.Series(nltk.pos_tag(nltk.WhitespaceTokenizer().tokenize(x)))
        else:
            s = pd.Series(nltk.pos_tag(nltk.word_tokenize(x)))
        return s
    
    df = df.sent_str.apply(word_tokenize).stack().to_frame().rename(columns = {0:'pos_tuple'})

    # grab info from tuple
    df['pos'] = df.pos_tuple.apply(lambda x: x[1])
    df['token_str'] = df.pos_tuple.apply(lambda x: x[0])
    if remove_pos_tuple:
        df = df.drop('pos_tuple', axis = 1)

    # add index
    df.index.names = OHCO

    return df

In [114]:
TOKEN = tokenize(DOC, ws = True)

TOKEN.head()

pos_tuple  \
work_id         part_num chap_num para_num sent_num token_num                     
modest_proposal 1        1        1        0        0                 (It, PRP)   
                                                    1                 (is, VBZ)   
                                                    2                   (a, DT)   
                                                    3          (melancholy, JJ)   
                                                    4              (object, NN)   

                                                               pos   token_str  
work_id         part_num chap_num para_num sent_num token_num                   
modest_proposal 1        1        1        0        0          PRP          It  
                                                    1          VBZ          is  
                                                    2           DT           a  
                                                    3           JJ  melancholy  
                                                    4           NN      object

## Build VOCAB

In [115]:
TOKEN['term_str'] = TOKEN['token_str'].str.lower().str.replace(r'[\W_]','',regex=True)
TOKEN.head()

pos_tuple  \
work_id         part_num chap_num para_num sent_num token_num                     
modest_proposal 1        1        1        0        0                 (It, PRP)   
                                                    1                 (is, VBZ)   
                                                    2                   (a, DT)   
                                                    3          (melancholy, JJ)   
                                                    4              (object, NN)   

                                                               pos  \
work_id         part_num chap_num para_num sent_num token_num        
modest_proposal 1        1        1        0        0          PRP   
                                                    1          VBZ   
                                                    2           DT   
                                                    3           JJ   
                                                    4           NN   

                                                                token_str  \
work_id         part_num chap_num para_num sent_num token_num               
modest_proposal 1        1        1        0        0                  It   
                                                    1                  is   
                                                    2                   a   
                                                    3          melancholy   
                                                    4              object   

                                                                 term_str  
work_id         part_num chap_num para_num sent_num token_num              
modest_proposal 1        1        1        0        0                  it  
                                                    1                  is  
                                                    2                   a  
                                                    3          melancholy  
                                                    4              object

In [116]:
VOCAB = (TOKEN.term_str.value_counts()
         .to_frame()
         .reset_index()
         .rename(columns = {'count':'n'})
         .sort_values('term_str')
         .reset_index(drop=True))
VOCAB.index.name = 'term_id'

In [117]:
VOCAB['num'] = VOCAB.term_str.str.match(r'\d+').astype('int')

VOCAB.sample(10)

,term_str,n,num
term_id,,,
5149,farmer,28,0
6688,hopped,1,0
4889,exantlation,1,0
11106,reality,2,0
6008,godmother,1,0
5943,gimlet,1,0
8206,louder,5,0
9144,no,730,0
5743,fronting,2,0


In [118]:
TOKEN = TOKEN.drop('pos_tuple',axis=1)

TOKEN.to_csv(f"{F2_path}/TOKEN.csv")
VOCAB.to_csv(f"{F2_path}/VOCAB.csv")